# Практика 2. Работа с LLM через API (GigaChat + LangChain)


## 0. Установка библиотек


In [1]:
import pandas as pd
from dotenv import load_dotenv
from langchain_gigachat.chat_models import GigaChat
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser

/Users/alekseu/Downloads/intro-profession/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


## 1. Импорт библиотек и подключение к GigaChat

In [2]:
import os
import json
import langchain_gigachat
import langchain_core
import pandas
from dotenv import load_dotenv

from langchain_gigachat.chat_models import GigaChat
from langchain_core.prompts import (
    ChatPromptTemplate,
    SystemMessagePromptTemplate,
    HumanMessagePromptTemplate,
)
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser

load_dotenv()
API_KEY = os.getenv("GIGA_KEY")

if not API_KEY:
    raise ValueError("Переменная окружения GIGA_KEY не задана.")

llm = GigaChat(
    credentials=API_KEY,
    model="GigaChat-2",
    verify_ssl_certs=False,
    temperature=0.2,
    max_tokens=1000,
)

print(llm.invoke("Привет! Ответь коротко.").content)

Привет. Коротко.


## 2. Выбор варианта и загрузка данных



In [3]:
VARIANT = "20"  

csv_path = f"rental_{VARIANT}.csv"

try:
    df = pd.read_csv(csv_path, sep=";", encoding="utf-8-sig")
except UnicodeDecodeError:
    df = pd.read_csv(csv_path, sep=";", encoding="cp1251")

required_columns = {"text", "amount"}
if not required_columns.issubset(df.columns):
    raise ValueError(f"В CSV должны быть столбцы {required_columns}. Найдены: {list(df.columns)}")

if len(df) != 15:
    raise ValueError(f"Ожидается 15 заявок, а найдено {len(df)}.")

df["amount"] = pd.to_numeric(df["amount"], errors="raise").astype(int)
print(f"Загружено заявок: {len(df)}")
display(df.head())

Загружено заявок: 15


,amount,text
0,6,Здравствуйте! Ищем жильё для 2 взрослых и 4 де...
1,2,"Добрый вечер! Ищем жилье на 2 человек, с 3 сен..."
2,4,Ищем жильё на 4 взрослых с 31.08.18 по 07.08.1...
3,3,"Здравствуйте, интересует жилье с 17.08 по 25.0..."
4,3,Добрый день. Ищем комфортное жильё в Лазаревс...


## 3. Задание 1 — базовый промпт

In [4]:
system_prompt = SystemMessagePromptTemplate.from_template(
    "Ты — эксперт по анализу текстовых заявок на аренду жилья. "
    "Определи общее количество людей, которые будут проживать."
)

user_prompt = HumanMessagePromptTemplate.from_template(
    "Заявка: {text}\n\n"
    "Извлеки общее количество проживающих. "
    "Верни только одно целое число, без пояснений."
)

chat_prompt = ChatPromptTemplate.from_messages([system_prompt, user_prompt])
chain = chat_prompt | llm | StrOutputParser()

print(chain.invoke({"text": "Семья из трех человек снимает квартиру"}))  # ожидается 3

3


## 4. Обработка 15 заявок и расчёт точности

In [5]:
def run_chain(chain_to_run, data, output_column):
    results = []
    for _, row in data.iterrows():
        try:
            answer = chain_to_run.invoke({"text": row["text"]})
            results.append(str(answer).strip())
        except Exception as error:
            results.append(f"ERROR: {error}")
    data[output_column] = results
    return data

def calc_accuracy(data, output_column):
    predicted = pd.to_numeric(data[output_column], errors="coerce")
    correct_mask = predicted == data["amount"]
    correct = int(correct_mask.sum())
    total = len(data)
    accuracy = correct / total if total else 0
    print(
        f"{output_column}: верных {correct} из {total}; "
        f"ошибок {total - correct}; точность {accuracy:.1%}"
    )
    return accuracy

df = run_chain(chain, df, "result")
acc_base = calc_accuracy(df, "result")
display(df[["text", "amount", "result"]])
df.to_csv("rental_with_results.csv", index=False, encoding="utf-8-sig")

result: верных 15 из 15; ошибок 0; точность 100.0%


,text,amount,result
0,Здравствуйте! Ищем жильё для 2 взрослых и 4 де...,6,6
1,"Добрый вечер! Ищем жилье на 2 человек, с 3 сен...",2,2
2,Ищем жильё на 4 взрослых с 31.08.18 по 07.08.1...,4,4
3,"Здравствуйте, интересует жилье с 17.08 по 25.0...",3,3
4,Добрый день. Ищем комфортное жильё в Лазаревс...,3,3
5,Ищем жильё в Лазаревское заезд 13 июля на 10 д...,4,4
6,Здравствуйте! Интересует жилье недорого 1-10 с...,4,4
7,Здравствуйте🤗\nИщем жильё в центре рядом с мор...,2,2
8,Здравствуйте! Ищем жильё с 15 по 23 августа 2...,3,3
9,Сниму жилье в июне у моря 2 взрослых до 1500 в...,2,2


## 5. Улучшение промпта — Few-shot


In [6]:
few_shot_system = SystemMessagePromptTemplate.from_template(
    """Ты — эксперт по анализу заявок на аренду жилья.
Определи ОБЩЕЕ количество проживающих: взрослых и детей вместе.

Правила:
- "семья с двумя детьми" = 4 человека (2 взрослых + 2 ребёнка);
- "семья с одним ребёнком" = 3 человека;
- "пара" = 2 человека;
- "один человек" = 1 человек;
- если прямо указано число людей, используй его;
- не путай количество комнат, спальных мест и людей.

Примеры:
Заявка: "Ищем жильё на троих" -> 3
Заявка: "Двое взрослых и четыре ребенка" -> 6
Заявка: "Одна взрослая и ребенок 5 лет" -> 2

Верни ТОЛЬКО целое число, без пояснений."""
)

few_shot_prompt = ChatPromptTemplate.from_messages([few_shot_system, user_prompt])
chain_fs = few_shot_prompt | llm | StrOutputParser()

df = run_chain(chain_fs, df, "result_fewshot")
acc_fs = calc_accuracy(df, "result_fewshot")
print(f"Изменение точности: {acc_base:.1%} -> {acc_fs:.1%}")
display(df[["text", "amount", "result", "result_fewshot"]])

result_fewshot: верных 15 из 15; ошибок 0; точность 100.0%
Изменение точности: 100.0% -> 100.0%


,text,amount,result,result_fewshot
0,Здравствуйте! Ищем жильё для 2 взрослых и 4 де...,6,6,6
1,"Добрый вечер! Ищем жилье на 2 человек, с 3 сен...",2,2,2
2,Ищем жильё на 4 взрослых с 31.08.18 по 07.08.1...,4,4,4
3,"Здравствуйте, интересует жилье с 17.08 по 25.0...",3,3,3
4,Добрый день. Ищем комфортное жильё в Лазаревс...,3,3,3
5,Ищем жильё в Лазаревское заезд 13 июля на 10 д...,4,4,4
6,Здравствуйте! Интересует жилье недорого 1-10 с...,4,4,4
7,Здравствуйте🤗\nИщем жильё в центре рядом с мор...,2,2,2
8,Здравствуйте! Ищем жильё с 15 по 23 августа 2...,3,3,3
9,Сниму жилье в июне у моря 2 взрослых до 1500 в...,2,2,2


### Заявки, на которых улучшенный промпт ошибся

In [7]:
pred_fs = pd.to_numeric(df["result_fewshot"], errors="coerce")
display(df[pred_fs != df["amount"]][["text", "amount", "result_fewshot"]])

,text,amount,result_fewshot


## 6. Задание 2 — извлечение дополнительных полей в JSON

Для анализа заявок используется структурированный JSON-ответ с количеством взрослых и детей, датой заезда, количеством ночей, ценой за сутки и дополнительными пожеланиями.

Для повышения качества извлечения применяется few-shot prompting. Примеры в промпте искусственные и намеренно не копируют тексты из тестового файла `rental_20.csv`, чтобы избежать утечки тестовых данных.

**Важно:** результаты JSON и метрики, сохранённые до этой редакции промпта, относятся к предыдущей версии. После изменений нужно заново выполнить ячейки подключения, извлечения, сохранения CSV и расчёта точности. Не используйте старые метрики как результат новой версии.

In [ ]:
import re
from datetime import datetime
from typing import Optional, Union
from pydantic import BaseModel, Field, StrictInt, StrictFloat, StrictStr

class RentalRequestFields(BaseModel):
    count: Optional[StrictInt] = Field(default=None, description="Общее количество проживающих")
    count_adults: Optional[StrictInt] = Field(default=None, description="Количество взрослых")
    count_children: Optional[StrictInt] = Field(default=None, description="Количество детей")
    start_date: Optional[StrictStr] = Field(default=None, description="Дата заезда в формате ДД.ММ")
    nights: Optional[StrictInt] = Field(default=None, description="Количество ночей или дней проживания")
    price_per_day: Optional[Union[StrictInt, StrictFloat]] = Field(default=None, description="Максимальная желаемая цена за сутки")
    remarks: Optional[StrictStr] = Field(default=None, description="Дополнительные пожелания к жилью")

json_parser = JsonOutputParser(pydantic_object=RentalRequestFields)

system_msg = """Ты — эксперт по извлечению данных из заявок на аренду жилья.
Извлекай только то, что подтверждается текстом. Не додумывай отсутствующие сведения.

ПРАВИЛА:
1. count — общее число проживающих, если его можно определить.
2. count_adults и count_children заполняй раздельно, только когда состав группы понятен.
   Если указано только «2 человека» или «на двоих», не угадывай возрастной состав:
   count_adults = null и count_children = null.
   Если явно сказано «одна женщина», «только женщина», «один мужчина»,
   «один взрослый» или аналогично, укажи одного взрослого и null для детей.
3. start_date — дата заезда строго в формате ДД.ММ.
   Указывай только день, который явно присутствует в заявке.
   Если указан только месяц или приблизительный период
   («в июне», «в начале июля», «на начало июля»,
   «примерно в июле»), верни null.
   Если указан диапазон дат, используй первую дату диапазона.
   Если год указан, не включай его в результат.
4. nights — количество ночей, рассчитываемое как разница
   между датой выезда и датой заезда.
   Не считай обе даты включительно.
   Примеры:
   - с 15.08 по 22.08 = 7 ночей;
   - с 01.09 по 10.09 = 9 ночей;
   - на 10 дней = 10.
   Если дата выезда раньше даты заезда в пределах одного
   указанного года или диапазон противоречивый,
   верни null. Не пытайся самостоятельно исправлять даты.
5. price_per_day — цена за сутки числом. Если дан диапазон цен, бери максимальную цену.
   Если цена не указана числом, верни null.
6. remarks — сохраняй явно указанные пожелания и ограничения:
   удобства, мебель и технику, парковку, домашних животных, запрет курения,
   район и расстояние до моря, тип/количество комнат и просьбы вроде
   «нужен один номер». Не добавляй факты, которых нет в заявке.
   Если пожеланий нет — null.
7. Не путай количество людей с количеством комнат, кроватей или спальных мест.
8. Верни все ключи из схемы. Неизвестные значения обозначай null.

ФОРМАТ ОТВЕТА:
{format_instructions}"""

few_shot_examples = [
    {
        "text": "Дом нужен для 2 взрослых и 3 детей, заезд 05.07, выезд 12.07; бюджет до 3500 рублей в сутки, можно с собакой.",
        "answer": {"count": 5, "count_adults": 2, "count_children": 3, "start_date": "05.07", "nights": 7, "price_per_day": 3500, "remarks": "можно с собакой"}
    },
    {
        "text": "Переезжаем: одна взрослая и двое детей, с 14.08 по 19.08; нужна детская кроватка.",
        "answer": {"count": 3, "count_adults": 1, "count_children": 2, "start_date": "14.08", "nights": 5, "price_per_day": None, "remarks": "детская кроватка"}
    },
    {
        "text": "Двое взрослых ищут жильё с 06.07 на 5 ночей; цена от 1800 до 2500 рублей в сутки, желательно балкон.",
        "answer": {"count": 2, "count_adults": 2, "count_children": None, "start_date": "06.07", "nights": 5, "price_per_day": 2500, "remarks": "балкон"}
    },
    {
        "text": "Для компании из пяти взрослых и младенца нужен дом с 20.08 по 25.08, нужна парковка.",
        "answer": {"count": 6, "count_adults": 5, "count_children": 1, "start_date": "20.08", "nights": 5, "price_per_day": None, "remarks": "парковка"}
    },
    {
        "text": "Нужна квартира на 4 человек, ориентировочно в сентябре, до 3000 рублей в сутки.",
        "answer": {"count": 4, "count_adults": None, "count_children": None, "start_date": None, "nights": None, "price_per_day": 3000, "remarks": None}
    },
    {
        "text": "Будет только одна женщина, жильё на 3 ночи, заезд 11.06.",
        "answer": {"count": 1, "count_adults": 1, "count_children": None, "start_date": "11.06", "nights": 3, "price_per_day": None, "remarks": None}
    }
]

# Защита от утечки тестовых данных: ни один текст примера не должен
# совпадать с одной из 15 заявок, на которых рассчитывается точность.
dataset_texts = set(df["text"].astype(str).str.strip())
example_texts = {item["text"].strip() for item in few_shot_examples}
overlap = dataset_texts.intersection(example_texts)
if overlap:
    raise ValueError(f"Few-shot-примеры пересекаются с тестовыми заявками: {overlap}")

few_shot_block = "\n\n".join(
    f'Заявка: "{item["text"]}"\nJSON: {json.dumps(item["answer"], ensure_ascii=False)}'
    for item in few_shot_examples
)
# Экранируем фигурные скобки JSON для шаблона ChatPromptTemplate.
few_shot_block = few_shot_block.replace("{", "{{").replace("}", "}}")

human_msg = (
    "Ниже приведены самостоятельные примеры формата и правил извлечения:\n\n"
    + few_shot_block
    + "\n\nТеперь обработай заявку:\n{text}\n\n"
    "Верни один JSON-объект со всеми ключами схемы, без текста до или после JSON."
)

extract_prompt = ChatPromptTemplate.from_messages([
    ("system", system_msg),
    ("human", human_msg),
]).partial(format_instructions=json_parser.get_format_instructions())

chain_json = extract_prompt | llm | json_parser

example_result = chain_json.invoke({"text": df.loc[0, "text"]})
# Дополнительно проверяем один демонстрационный ответ схемой Pydantic.
example_result = RentalRequestFields.parse_obj(example_result).dict()
print(json.dumps(example_result, ensure_ascii=False, indent=2))

### 6.1. Обработка всех 15 заявок и сохранение ответов

In [ ]:
expected_json_keys = [
    "count", "count_adults", "count_children",
    "start_date", "nights", "price_per_day", "remarks"
]

rows = []
for idx, row in df.iterrows():
    try:
        result = chain_json.invoke({"text": row["text"]})

        if not isinstance(result, dict):
            raise ValueError("Ответ модели не является JSON-объектом")

        # Сначала проверяем комплектность JSON, затем типы и значения через Pydantic.
        missing_keys = [key for key in expected_json_keys if key not in result]
        if missing_keys:
            raise ValueError(f"В JSON отсутствуют ключи: {missing_keys}")

        result = {key: result[key] for key in expected_json_keys}
        validated_result = RentalRequestFields.parse_obj(result)
        result = validated_result.dict()

        # Проверяем логические ограничения, а не только JSON-типы.
        if result["count"] is not None and result["count"] < 1:
            raise ValueError("count должен быть положительным числом")
        for field in ("count_adults", "count_children"):
            if result[field] is not None and result[field] < 0:
                raise ValueError(f"{field} не может быть отрицательным")
        if result["nights"] is not None and result["nights"] < 1:
            raise ValueError("nights должен быть положительным числом")
        if result["price_per_day"] is not None and result["price_per_day"] < 0:
            raise ValueError("price_per_day не может быть отрицательной")

        adults = result["count_adults"]
        children = result["count_children"]
        total_people = result["count"]
        if (
            total_people is not None
            and adults is not None
            and children is not None
            and total_people != adults + children
        ):
            raise ValueError("count не совпадает с суммой count_adults и count_children")

        start_date = result["start_date"]
        if start_date is not None:
            if re.fullmatch(r"\\d{2}\\.\\d{2}", start_date) is None:
                raise ValueError("start_date должен иметь формат ДД.ММ")
            try:
                datetime.strptime(f"{start_date}.2000", "%d.%m.%Y")
            except ValueError as date_error:
                raise ValueError("start_date содержит некорректную календарную дату") from date_error

        result["parse_error"] = None

    except Exception as error:
        result = {key: None for key in expected_json_keys}
        result["parse_error"] = str(error)

    result["idx"] = idx
    rows.append(result)

df_json = pd.DataFrame(rows).set_index("idx")

if len(df_json) != 15:
    raise ValueError(f"Ожидалось 15 результатов, получено {len(df_json)}.")
if df_json.index.has_duplicates:
    raise ValueError("В результатах обнаружены повторяющиеся индексы заявок.")

df_json.to_csv("llm_extracted.csv", encoding="utf-8-sig")
print(f"Сохранено результатов: {len(df_json)}")
print(f"Ошибок разбора или Pydantic-валидации: {df_json['parse_error'].notna().sum()}")
display(df_json)

## 7. Эталонная разметка

Файл `gold_manual.csv` содержит эталонные значения, определённые по текстам 15 заявок. Он используется для сравнения ответов модели с проверенными значениями.

Для расчёта точности используются поля `count_adults`, `count_children`, `start_date`, `nights` и `price_per_day`. Даты приводятся к единому формату `ДД.ММ`.

### 7.1. Расчёт точности по дополнительным полям

Пустое значение в эталоне и `null` у модели считаются совпадением: это означает, что в заявке действительно нет информации для данного поля. Если в эталоне значение есть, а модель вернула `null` (или наоборот), это ошибка.

В среднюю точность входят пять проверяемых колонок: взрослые, дети, дата заезда, ночи и цена. `remarks` проверяется отдельно вручную и в среднюю точность не входит.

In [ ]:
import re

gold = pd.read_csv(
    "gold_manual.csv",
    index_col="idx",
    encoding="utf-8-sig",
    dtype={"start_date": "string"}
)

fields = [
    "count_adults",
    "count_children",
    "start_date",
    "nights",
    "price_per_day",
]
numeric_fields = {"count_adults", "count_children", "nights", "price_per_day"}

# Не допускаем тихое сравнение неполных или повторяющихся наборов строк.
if gold.index.has_duplicates:
    raise ValueError("В gold_manual.csv обнаружены повторяющиеся индексы.")
if set(gold.index) != set(df_json.index):
    raise ValueError("Набор индексов в ручной разметке и ответах модели не совпадает.")
gold = gold.reindex(df_json.index)

def normalize_value(value, field):
    # Приводим пустые ячейки, NaN и текстовые null к одному значению.
    if pd.isna(value):
        return None

    if isinstance(value, str):
        value = value.strip()
        if value.lower() in ("", "nan", "none", "null"):
            return None

    if field in numeric_fields:
        try:
            return float(value)
        except (TypeError, ValueError):
            return str(value).strip()

    if field == "start_date":
        value = str(value).strip()
        match = re.fullmatch(
            r"(\d{1,2})\.(\d{1,2})(?:\.(?:\d{2}|\d{4}))?",
            value
        )
        if match:
            day, month = map(int, match.groups())
            return f"{day:02d}.{month:02d}"
        return value

    return value

accuracies = {}
mismatches = []

for field in fields:
    if field not in gold.columns or field not in df_json.columns:
        raise ValueError(f"Не найден столбец {field}. Проверьте CSV и JSON-ответы.")

    manual_values = gold[field]
    model_values = df_json[field]

    matches = []
    known_matches = []
    known_expected_count = 0
    extracted_known_count = 0

    for idx in df_json.index:
        expected = normalize_value(manual_values.loc[idx], field)
        predicted = normalize_value(model_values.loc[idx], field)

        # Ошибка разбора/валидации означает, что ответа для этой заявки нет.
        # Даже если эталонное поле пустое, такую строку нельзя считать совпадением.
        parse_error_value = (
            df_json.loc[idx, "parse_error"]
            if "parse_error" in df_json.columns else None
        )
        parse_failed = (
            pd.notna(parse_error_value)
            and str(parse_error_value).strip().lower() not in ("", "none", "nan", "null")
        )

        # Пустое значение считается верным только при успешном разборе JSON.
        is_match = (expected == predicted) and not parse_failed
        matches.append(is_match)

        # Дополнительно считаем качество на строках, где эталонное значение известно,
        # чтобы отдельно видеть ошибки извлечения непустых значений.
        if expected is not None:
            known_expected_count += 1
            known_matches.append(is_match)
            if predicted is not None:
                extracted_known_count += 1

        if not is_match:
            mismatches.append({
                "idx": idx,
                "field": field,
                "expected": expected,
                "predicted": predicted,
                "parse_error": parse_error_value if parse_failed else None,
            })

    correct = sum(matches)
    total = len(matches)
    accuracy = correct / total if total else 0.0
    accuracies[field] = accuracy

    known_accuracy = (
        sum(known_matches) / known_expected_count
        if known_expected_count else float("nan")
    )
    extraction_coverage = (
        extracted_known_count / known_expected_count
        if known_expected_count else float("nan")
    )

    print(
        f"{field}: {correct} из {total} = {accuracy:.1%}; "
        f"точность на непустых эталонных значениях: {known_accuracy:.1%}; "
        f"полнота извлечения непустых значений: {extraction_coverage:.1%}"
    )

average_accuracy = sum(accuracies.values()) / len(accuracies)
print(f"Средняя точность по пяти полям: {average_accuracy:.1%}")
print(f"Ошибок разбора или Pydantic-валидации: {df_json['parse_error'].notna().sum()} из {len(df_json)}")

if mismatches:
    print("\nОшибки, которые стоит разобрать:")
    display(pd.DataFrame(mismatches))
else:
    print("\nОшибок по проверяемым полям нет.")